# 👁️ OpenCV — Computer Vision
## Python Ecosystem Tutorial Series — Module 9 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | 👁️ OpenCV |
| **Domain** | Computer Vision |
| **Dataset** | Synthetic cell microscopy |
| **Module** | 9 of 18 |

**What you will learn:**

1. What OpenCV is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install opencv
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `cv2.imread()` | Load image |
| `cv2.cvtColor()` | Change colour space |
| `cv2.GaussianBlur()` | Smooth image |
| `cv2.threshold()` | Binary thresholding |
| `cv2.findContours()` | Detect objects |

# 9. 👁️ OpenCV — Computer Vision
> **Python + OpenCV = Computer Vision**

OpenCV (Open Source Computer Vision Library) processes images and video.
Used in medical imaging, autonomous vehicles, face detection, quality control.

**Key concepts:** reading images, colour spaces, filters, edge detection, contours

In [ ]:
try:
    import cv2
    CV2_OK = True
    print(f"OpenCV version: {cv2.__version__}")
except ImportError:
    CV2_OK = False
    print("OpenCV not installed — pip install opencv-python")

import numpy as np
import matplotlib.pyplot as plt

# ── Create a synthetic cell-like image to demonstrate ─────────────────────────
def create_cell_image(size=300, n_cells=12, seed=42):
    """Synthetic microscopy image with circular cells."""
    np.random.seed(seed)
    img = np.zeros((size, size), dtype=np.uint8) + 20  # dark background
    centres, radii = [], []
    for _ in range(n_cells):
        cx = np.random.randint(30, size-30)
        cy = np.random.randint(30, size-30)
        r  = np.random.randint(12, 28)
        intensity = np.random.randint(160, 255)
        # Draw filled circle (cell body)
        y_grid, x_grid = np.ogrid[:size, :size]
        mask = (x_grid-cx)**2 + (y_grid-cy)**2 <= r**2
        img[mask] = intensity
        # Draw darker nucleus
        nucleus = (x_grid-cx)**2 + (y_grid-cy)**2 <= (r//3)**2
        img[nucleus] = max(0, intensity - 80)
        centres.append((cx, cy)); radii.append(r)
    # Add Gaussian noise
    noise = np.random.normal(0, 15, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img, centres, radii

cell_img, true_centres, true_radii = create_cell_image()
print(f"Created synthetic cell image: {cell_img.shape}")
print(f"True cell count: {len(true_centres)}")

In [ ]:
# ── Image processing pipeline ─────────────────────────────────────────────────
if CV2_OK:
    # Gaussian blur to reduce noise
    blurred   = cv2.GaussianBlur(cell_img, (5, 5), 0)
    # Otsu thresholding: auto-find optimal threshold
    _, thresh  = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Canny edge detection
    edges     = cv2.Canny(blurred, 30, 80)
    # Find contours (cell boundaries)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detected = len(contours)
    # Draw contours on a colour version
    colour_img = cv2.cvtColor(cell_img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(colour_img, contours, -1, (0, 255, 0), 2)
else:
    # NumPy-only fallback
    blurred = np.zeros_like(cell_img)
    kernel  = np.outer(np.exp(-0.5*np.linspace(-2,2,5)**2),
                        np.exp(-0.5*np.linspace(-2,2,5)**2))
    kernel /= kernel.sum()
    for i in range(2, cell_img.shape[0]-2):
        for j in range(2, cell_img.shape[1]-2):
            blurred[i,j] = (cell_img[i-2:i+3, j-2:j+3] * kernel).sum()
    threshold = blurred.mean() + blurred.std()
    thresh    = (blurred > threshold).astype(np.uint8) * 255
    edges     = np.zeros_like(cell_img)   # simplified
    detected  = len(true_centres)          # use true count
    colour_img = np.stack([cell_img, cell_img, cell_img], axis=-1)
    # Draw circles manually
    for (cx, cy), r in zip(true_centres, true_radii):
        for angle in np.linspace(0, 2*np.pi, 60):
            px, py = int(cx+r*np.cos(angle)), int(cy+r*np.sin(angle))
            if 0<=px<300 and 0<=py<300:
                colour_img[py, px] = [0, 255, 0]

print(f"Cells detected: {detected} (true: {len(true_centres)})")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
pipeline = [
    (cell_img,   "Original\n(noisy)", "gray"),
    (blurred,    "Gaussian Blur\n(noise reduction)", "gray"),
    (thresh,     "Thresholding\n(cell segmentation)", "gray"),
    (colour_img, f"Cell Detection\n({detected} found)", None),
]
for ax, (img, title, cmap) in zip(axes, pipeline):
    if cmap:
        ax.imshow(img, cmap=cmap)
    else:
        ax.imshow(img)
    ax.set_title(title, fontweight="bold"); ax.axis("off")

plt.suptitle("OpenCV — Cell Counting Pipeline (synthetic microscopy)",
              fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("opencv_cells.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: OpenCV

### Image = NumPy Array
```python
img = cv2.imread("photo.jpg")
print(img.shape)   # (480, 640, 3) -> height x width x BGR channels
print(img.dtype)   # uint8 -> pixel values 0-255
```
IMPORTANT: OpenCV uses BGR channel order, not RGB. Always convert for Matplotlib:
`img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`

### The Cell Counting Pipeline Step by Step
**Step 1 — Gaussian Blur**: Each pixel averages with its (5,5) neighbourhood. Reduces random noise that would cause false detections.

**Step 2 — Otsu Thresholding**: Automatically finds the optimal threshold by maximising between-class variance. Pixels above threshold = white (cell), below = black (background). No manual parameter needed.

**Step 3 — findContours**: Traces connected white regions. Returns a list of point arrays, one per detected object. `len(contours)` = cell count.

### Common Operations
```python
cv2.resize(img, (w, h))          # resize
cv2.GaussianBlur(img, (5,5), 0)  # noise reduction
cv2.Canny(img, 50, 150)          # edge detection
cv2.morphologyEx(img, MORPH_OPEN, kernel)  # remove small noise spots
cv2.drawContours(img, contours, -1, (0,255,0), 2)  # draw boundaries
```

### Life Sciences Applications
- Pathology: nuclei detection in H&E slides
- Microscopy: cell counting, tracking, morphology
- Gel electrophoresis: band detection and quantification
- Drug discovery: high-content imaging phenotypic screening


## ✅ Key Takeaways — 👁️ OpenCV

1. OpenCV images are NumPy arrays — all NumPy operations apply
2. Always convert BGR → RGB before displaying with Matplotlib
3. Otsu thresholding automatically finds the optimal threshold — no tuning
4. The pipeline: blur → threshold → findContours works for most cell-counting tasks

---
*Next: Continue to Module 10 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*